# Еще признаки

В прошлом ноутбуке CatBoost дошел примерно до 0.571

Проверим можно ли выжать больше просто за счет нормального описания поведения куки

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from metric import precision_at_recall

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /Users/an.m.titova/Documents/bot_detection_case


In [2]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

events = pd.read_csv(
    DATA_DIR / "events.csv.gz",
    parse_dates=["event_ts"],
)

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)

assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique
assert train["target"].isin([0, 1]).all()


train: (11091, 5)
test: (4909, 4)
events: (328905, 14)


In [3]:
def filter_events_by_window(events, meta):
    events_with_window = events.merge(
        meta[
            [
                "cookie_id",
                "window_start_ts",
                "window_end_ts",
            ]
        ],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )

    mask = (
        (events_with_window["event_ts"] >= events_with_window["window_start_ts"])
        &
        (events_with_window["event_ts"] < events_with_window["window_end_ts"])
    )

    return events_with_window.loc[
        mask,
        events.columns
    ].copy()


events_train = filter_events_by_window(events, train)
events_test = filter_events_by_window(events, test)

print("Train events:", len(events_train))
print("Test events:", len(events_test))


Train events: 198436
Test events: 89690


In [ ]:
def prepare_events(events):
    result = events.copy()

    result["platform_clean"] = (
        result["platform"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return result


events_train = prepare_events(events_train)
events_test = prepare_events(events_test)

EVENT_TYPES = sorted(
    events_train["event_name"]
    .dropna()
    .unique()
)

PLATFORM_TYPES = sorted(
    events_train["platform_clean"]
    .dropna()
    .unique()
)

print("EVENT_TYPES:", EVENT_TYPES)
print("PLATFORM_TYPES:", PLATFORM_TYPES)


EVENT_TYPES: ['contact_chat_open', 'contact_message_sent', 'contact_phone_show', 'favorite_add', 'item_view', 'login', 'photo_swipe', 'search_results_view', 'seller_page_view']
PLATFORM_TYPES: ['android', 'desktop', 'ios', 'iphone', 'web']


In [ ]:
def build_features(meta, events, event_types, platform_types):
    events = events.copy()

    events["platform_clean"] = (
        events["platform"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    features = meta[
        [
            "cookie_id",
            "cookie_created_at",
            "window_start_ts",
            "window_end_ts",
        ]
    ].copy()

   

    original_event_columns = [
        col for col in events.columns
        if col != "platform_clean"
    ]

    duplicate_mask = events.duplicated(
        subset=original_event_columns,
        keep="first",
    )


    duplicate_features = (
        events
        .assign(is_exact_duplicate=duplicate_mask.astype(int))
        .groupby("cookie_id")
        .agg(
            exact_duplicate_count=("is_exact_duplicate", "sum"),
            n_events_raw=("is_exact_duplicate", "size"),
        )
        .reset_index()
    )

    duplicate_features["exact_duplicate_share"] = (
        duplicate_features["exact_duplicate_count"]
        / duplicate_features["n_events_raw"].replace(0, np.nan)
    )

    features = features.merge(
        duplicate_features[
            [
                "cookie_id",
                "n_events_raw",
                "exact_duplicate_count",
                "exact_duplicate_share",
            ]
        ],
        on="cookie_id",
        how="left",
    )

    events = events.loc[~duplicate_mask].copy()



    features["cookie_age_days"] = (
        features["window_start_ts"]
        - features["cookie_created_at"]
    ).dt.total_seconds() / (24 * 60 * 60)


    event_counts = pd.crosstab(
        events["cookie_id"],
        events["event_name"],
    )

    event_counts = (
        event_counts
        .reindex(columns=event_types, fill_value=0)
        .add_prefix("cnt_event_")
        .reset_index()
    )

    features = features.merge(
        event_counts,
        on="cookie_id",
        how="left",
    )

    count_cols = [
        f"cnt_event_{event_type}"
        for event_type in event_types
    ]

    features[count_cols] = (
        features[count_cols]
        .fillna(0)
    )

    features["n_events"] = (
        features[count_cols]
        .sum(axis=1)
    )

    features["n_event_types"] = (
        (features[count_cols] > 0)
        .sum(axis=1)
    )


    for col in count_cols:
        event_name = col.replace("cnt_event_", "")

        features[f"share_event_{event_name}"] = (
            features[col]
            / features["n_events"].replace(0, np.nan)
        )

        features[f"has_event_{event_name}"] = (
            features[col] > 0
        ).astype(int)


    diversity = (
        events
        .groupby("cookie_id")
        .agg(
            n_unique_items=("item_id", "nunique"),
            n_unique_categories=("item_category", "nunique"),
            n_unique_locations=("item_location", "nunique"),
            n_unique_seller_types=("seller_type", "nunique"),
            n_unique_queries=("search_query", "nunique"),
            n_unique_platforms=("platform_clean", "nunique"),
            n_unique_user_agents=("user_agent", "nunique"),
        )
        .reset_index()
    )

    features = features.merge(
        diversity,
        on="cookie_id",
        how="left",
    )

    diversity_cols = [
        "n_unique_items",
        "n_unique_categories",
        "n_unique_locations",
        "n_unique_seller_types",
        "n_unique_queries",
        "n_unique_platforms",
        "n_unique_user_agents",
    ]

    features[diversity_cols] = (
        features[diversity_cols]
        .fillna(0)
    )

   

    def entropy_top_share(frame, column, prefix):
        tmp = frame.loc[
            frame[column].notna(),
            ["cookie_id", column],
        ].copy()

        if tmp.empty:
            return pd.DataFrame(
                columns=[
                    "cookie_id",
                    f"{prefix}_entropy",
                    f"{prefix}_top_share",
                ]
            )

        counts = (
            tmp
            .groupby(["cookie_id", column])
            .size()
            .rename("count")
            .reset_index()
        )

        totals = (
            counts
            .groupby("cookie_id")["count"]
            .transform("sum")
        )

        counts["share"] = (
            counts["count"] / totals
        )

        entropy = (
            counts
            .assign(
                entropy_part=lambda x:
                    -(x["share"] * np.log(x["share"]))
            )
            .groupby("cookie_id")
            .agg(
                entropy_value=("entropy_part", "sum"),
                top_share_value=("share", "max"),
            )
            .reset_index()
            .rename(
                columns={
                    "entropy_value": f"{prefix}_entropy",
                    "top_share_value": f"{prefix}_top_share",
                }
            )
        )

        return entropy

    entropy_columns = {
        "platform_clean": "platform",
        "user_agent": "user_agent",
        "item_id": "item",
        "item_category": "category",
        "item_location": "location",
        "seller_type": "seller",
        "search_query": "query",
        "search_page": "search_page",
    }

    for column, prefix in entropy_columns.items():
        stats = entropy_top_share(
            events,
            column,
            prefix,
        )

        features = features.merge(
            stats,
            on="cookie_id",
            how="left",
        )


    search_features = (
        events
        .groupby("cookie_id")
        .agg(
            mean_search_page=("search_page", "mean"),
            max_search_page=("search_page", "max"),
        )
        .reset_index()
    )

    features = features.merge(
        search_features,
        on="cookie_id",
        how="left",
    )


    pointer = events[
        [
            "cookie_id",
            "pointer_x",
            "pointer_y",
        ]
    ].copy()

    pointer["has_pointer"] = (
        pointer["pointer_x"].notna()
        & pointer["pointer_y"].notna()
    )

    pointer_features = (
        pointer
        .groupby("cookie_id")
        .agg(
            n_pointer_events=("has_pointer", "sum"),
            share_pointer_events=("has_pointer", "mean"),
        )
        .reset_index()
    )

    features = features.merge(
        pointer_features,
        on="cookie_id",
        how="left",
    )

    features[
        [
            "n_pointer_events",
            "share_pointer_events",
        ]
    ] = features[
        [
            "n_pointer_events",
            "share_pointer_events",
        ]
    ].fillna(0)


    platform_counts = pd.crosstab(
        events["cookie_id"],
        events["platform_clean"],
    )

    platform_counts = (
        platform_counts
        .reindex(columns=platform_types, fill_value=0)
        .add_prefix("cnt_platform_")
        .reset_index()
    )

    features = features.merge(
        platform_counts,
        on="cookie_id",
        how="left",
    )

    platform_cols = [
        f"cnt_platform_{platform}"
        for platform in platform_types
    ]

    features[platform_cols] = (
        features[platform_cols]
        .fillna(0)
    )

    for col in platform_cols:
        platform = col.replace("cnt_platform_", "")

        features[f"share_platform_{platform}"] = (
            features[col]
            / features["n_events"].replace(0, np.nan)
        )


    headless = events[
        ["cookie_id", "user_agent"]
    ].copy()

    headless["is_headless"] = (
        headless["user_agent"]
        .fillna("")
        .str.lower()
        .str.contains(
            "headlesschrome",
            regex=False,
        )
        .astype(int)
    )

    headless_features = (
        headless
        .groupby("cookie_id")
        .agg(
            n_headless_events=("is_headless", "sum"),
            share_headless_events=("is_headless", "mean"),
            has_headless=("is_headless", "max"),
        )
        .reset_index()
    )

    features = features.merge(
        headless_features,
        on="cookie_id",
        how="left",
    )

    features[
        [
            "n_headless_events",
            "share_headless_events",
            "has_headless",
        ]
    ] = features[
        [
            "n_headless_events",
            "share_headless_events",
            "has_headless",
        ]
    ].fillna(0)


    events_sorted = (
        events
        .sort_values(["cookie_id", "event_ts"])
        .copy()
    )

    events_sorted["time_diff_sec"] = (
        events_sorted
        .groupby("cookie_id")["event_ts"]
        .diff()
        .dt.total_seconds()
    )


    time_features = (
        events_sorted
        .groupby("cookie_id")["time_diff_sec"]
        .agg(
            mean_gap_sec="mean",
            median_gap_sec="median",
            min_gap_sec="min",
            max_gap_sec="max",
            std_gap_sec="std",
        )
        .reset_index()
    )

    valid_gaps = events_sorted[
        events_sorted["time_diff_sec"].notna()
    ].copy()


    gap_quantiles = (
        valid_gaps
        .groupby("cookie_id")["time_diff_sec"]
        .quantile([0.10, 0.25, 0.75, 0.90])
        .unstack()
        .reset_index()
        .rename(
            columns={
                0.10: "gap_q10_sec",
                0.25: "gap_q25_sec",
                0.75: "gap_q75_sec",
                0.90: "gap_q90_sec",
            }
        )
    )

    time_features = time_features.merge(
        gap_quantiles,
        on="cookie_id",
        how="left",
    )

    time_features["gap_cv"] = (
        time_features["std_gap_sec"]
        / time_features["mean_gap_sec"].replace(0, np.nan)
    )


    for threshold in [1, 2, 5, 10, 30, 60, 300]:
        gap_share = (
            valid_gaps
            .assign(
                is_fast=lambda x:
                    x["time_diff_sec"] <= threshold
            )
            .groupby("cookie_id")["is_fast"]
            .mean()
            .rename(f"share_gap_le_{threshold}s")
            .reset_index()
        )

        time_features = time_features.merge(
            gap_share,
            on="cookie_id",
            how="left",
        )


    activity_span = (
        events_sorted
        .groupby("cookie_id")["event_ts"]
        .agg(
            first_event_ts="min",
            last_event_ts="max",
        )
        .reset_index()
    )

    activity_span = activity_span.merge(
        meta[
            [
                "cookie_id",
                "window_start_ts",
                "window_end_ts",
            ]
        ],
        on="cookie_id",
        how="left",
    )

    activity_span["active_span_sec"] = (
        activity_span["last_event_ts"]
        - activity_span["first_event_ts"]
    ).dt.total_seconds()

    activity_span["first_event_delay_sec"] = (
        activity_span["first_event_ts"]
        - activity_span["window_start_ts"]
    ).dt.total_seconds()

    activity_span["last_event_to_window_end_sec"] = (
        activity_span["window_end_ts"]
        - activity_span["last_event_ts"]
    ).dt.total_seconds()

    time_features = time_features.merge(
        activity_span[
            [
                "cookie_id",
                "active_span_sec",
                "first_event_delay_sec",
                "last_event_to_window_end_sec",
            ]
        ],
        on="cookie_id",
        how="left",
    )


    events_sorted["event_minute"] = (
        events_sorted["event_ts"].dt.floor("min")
    )

    minute_counts = (
        events_sorted
        .groupby(["cookie_id", "event_minute"])
        .size()
        .rename("events_in_minute")
        .reset_index()
    )

    minute_features = (
        minute_counts
        .groupby("cookie_id")["events_in_minute"]
        .agg(
            events_per_minute_mean="mean",
            events_per_minute_std="std",
            events_per_minute_max="max",
            active_minutes="count",
        )
        .reset_index()
    )

    events_sorted["event_hour_bucket"] = (
        events_sorted["event_ts"].dt.floor("h")
    )

    hour_counts = (
        events_sorted
        .groupby(["cookie_id", "event_hour_bucket"])
        .size()
        .rename("events_in_hour")
        .reset_index()
    )

    hour_features = (
        hour_counts
        .groupby("cookie_id")["events_in_hour"]
        .agg(
            events_per_hour_mean="mean",
            events_per_hour_std="std",
            events_per_hour_max="max",
            active_hours="count",
        )
        .reset_index()
    )

    time_features = (
        time_features
        .merge(
            minute_features,
            on="cookie_id",
            how="left",
        )
        .merge(
            hour_features,
            on="cookie_id",
            how="left",
        )
    )


    events_sorted["previous_event_name"] = (
        events_sorted
        .groupby("cookie_id")["event_name"]
        .shift()
    )

    transitions = events_sorted[
        events_sorted["previous_event_name"].notna()
    ].copy()

    transitions["same_event_transition"] = (
        transitions["previous_event_name"]
        == transitions["event_name"]
    ).astype(int)

    transitions["transition"] = (
        transitions["previous_event_name"].astype(str)
        + ">"
        + transitions["event_name"].astype(str)
    )

    transition_features = (
        transitions
        .groupby("cookie_id")
        .agg(
            same_event_transition_share=(
                "same_event_transition",
                "mean",
            ),
            transition_nunique=(
                "transition",
                "nunique",
            ),
        )
        .reset_index()
    )

    time_features = time_features.merge(
        transition_features,
        on="cookie_id",
        how="left",
    )

    features = features.merge(
        time_features,
        on="cookie_id",
        how="left",
    )

    features["events_per_active_hour"] = (
        features["n_events"]
        / (
            features["active_span_sec"] / 3600
        ).clip(lower=1 / 3600)
    )


    def safe_ratio(num, den):
        return num / den.replace(0, np.nan)

    if "cnt_event_item_view" in features.columns:
        features["unique_items_per_item_view"] = safe_ratio(
            features["n_unique_items"],
            features["cnt_event_item_view"],
        )

    if (
        "cnt_event_photo_swipe" in features.columns
        and "cnt_event_item_view" in features.columns
    ):
        features["photo_swipe_per_item_view"] = safe_ratio(
            features["cnt_event_photo_swipe"],
            features["cnt_event_item_view"],
        )

    if (
        "cnt_event_favorite_add" in features.columns
        and "cnt_event_item_view" in features.columns
    ):
        features["favorite_per_item_view"] = safe_ratio(
            features["cnt_event_favorite_add"],
            features["cnt_event_item_view"],
        )

   
    features = features.replace([np.inf, -np.inf], np.nan)

    assert features["cookie_id"].is_unique

    assert not any(
        col.endswith("_x") or col.endswith("_y")
        for col in features.columns
    )

    return features

Что добавила:

- точные дубли
- больше статистик по gap
- активность по минутам и часам
- переходы между событиями
- entropy и top share

С дублями делаю так, сначала считаю их количество, потом убираю из остальных агрегатов. Иначе они будут искусственно раздувать активность

In [6]:
train_features = build_features(
    train,
    events_train,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

test_features = build_features(
    test,
    events_test,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

train_features = train_features.merge(
    train[["cookie_id", "target"]],
    on="cookie_id",
    how="left",
)

assert len(train_features) == len(train)
assert len(test_features) == len(test)

assert train_features["cookie_id"].is_unique
assert test_features["cookie_id"].is_unique

assert not any(
    col.endswith("_x") or col.endswith("_y")
    for col in train_features.columns
)

train_feature_columns = [
    col
    for col in train_features.columns
    if col != "target"
]

assert train_feature_columns == test_features.columns.tolist()

print("Train/test feature schema: OK")
print("Train features:", train_features.shape)
print("Test features:", test_features.shape)


Train/test feature schema: OK
Train features: (11091, 112)
Test features: (4909, 111)


In [7]:
train_features.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,n_events_raw,exact_duplicate_count,exact_duplicate_share,cookie_age_days,cnt_event_contact_chat_open,cnt_event_contact_message_sent,...,events_per_hour_std,events_per_hour_max,active_hours,same_event_transition_share,transition_nunique,events_per_active_hour,unique_items_per_item_view,photo_swipe_per_item_view,favorite_per_item_view,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,7,0,0.000000,135.603692,0,0,...,NaN,7,1,0.333333,5.0,150.898204,1.333333,0.333333,0.000000,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,41,0,0.000000,194.576111,0,0,...,3.720119,12,8,0.425000,18.0,3.738981,0.863636,0.181818,0.045455,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,36,0,0.000000,32.994421,1,1,...,2.609506,10,7,0.142857,23.0,3.551561,2.714286,0.428571,0.428571,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,21,0,0.000000,0.546759,1,0,...,3.605551,10,3,0.450000,9.0,4.389989,0.714286,0.000000,0.000000,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,29,1,0.034483,94.837095,0,0,...,7.127412,18,5,0.259259,16.0,5.278040,1.333333,0.333333,0.166667,0


Сравниваю на том же последнем фолде 17-19 апреля, чтобы увидеть разницу с предыдущей версией

In [8]:
VALID_START = pd.Timestamp("2026-04-17")

is_valid = (
    train_features["window_start_ts"]
    >= VALID_START
)

train_part = train_features.loc[~is_valid].copy()
valid_part = train_features.loc[is_valid].copy()

NON_FEATURE_COLS = [
    "cookie_id",
    "target",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]

feature_cols = [
    col
    for col in train_features.columns
    if col not in NON_FEATURE_COLS
]

X_train = train_part[feature_cols]
y_train = train_part["target"]

X_valid = valid_part[feature_cols]
y_valid = valid_part["target"]

forbidden = set(NON_FEATURE_COLS)
assert forbidden.isdisjoint(feature_cols)

print("Train part:", len(train_part))
print("Valid part:", len(valid_part))
print("Признаков модели:", len(feature_cols))


Train part: 9140
Valid part: 1951
Признаков модели: 107


In [9]:
cat_rich = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False,
)

cat_rich.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=80,
    verbose=False,
)

score_cat_rich = cat_rich.predict_proba(X_valid)[:, 1]


hgb_rich = HistGradientBoostingClassifier(
    learning_rate=0.045,
    max_iter=350,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

hgb_rich.fit(
    X_train,
    y_train,
)

score_hgb_rich = hgb_rich.predict_proba(X_valid)[:, 1]


In [10]:
rich_results = pd.DataFrame(
    [
        {
            "model": "CatBoost",
            "p_at_r_70": precision_at_recall(
                y_valid,
                score_cat_rich,
            ),
            "roc_auc": roc_auc_score(
                y_valid,
                score_cat_rich,
            ),
            "pr_auc": average_precision_score(
                y_valid,
                score_cat_rich,
            ),
        },
        {
            "model": "HistGradientBoosting",
            "p_at_r_70": precision_at_recall(
                y_valid,
                score_hgb_rich,
            ),
            "roc_auc": roc_auc_score(
                y_valid,
                score_hgb_rich,
            ),
            "pr_auc": average_precision_score(
                y_valid,
                score_hgb_rich,
            ),
        },
    ]
).sort_values(
    "p_at_r_70",
    ascending=False,
)

rich_results


,model,p_at_r_70,roc_auc,pr_auc
0,CatBoost,0.634831,0.897669,0.725027
1,HistGradientBoosting,0.541063,0.892748,0.717647


CatBoost вырос примерно с 0.571 до 0.635. HGB на этих признаках хуже, поэтому пока CatBoost оставляю основным

In [11]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": cat_rich.get_feature_importance(),
}).sort_values(
    "importance",
    ascending=False,
)

feature_importance.head(25).reset_index(drop=True)


,feature,importance
0,share_gap_le_60s,4.168500
1,n_unique_categories,3.976771
2,unique_items_per_item_view,3.587962
3,category_entropy,3.350988
4,gap_q75_sec,3.233692
5,mean_search_page,3.122759
6,events_per_minute_mean,2.857325
7,share_gap_le_30s,2.750721
8,gap_q25_sec,2.739808
9,cookie_age_days,2.619790


В importance появились `category_entropy`, `location_entropy`, `item_top_share` и дополнительные gap признаки

In [ ]:
def evaluate_temporal_fold(
    valid_start,
    valid_end,
    features,
    feature_cols,
    model_name,
):
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    train_mask = (
        features["window_start_ts"] < valid_start
    )

    valid_mask = (
        (features["window_start_ts"] >= valid_start)
        &
        (features["window_start_ts"] <= valid_end)
    )

    X_train_fold = features.loc[
        train_mask,
        feature_cols,
    ]

    y_train_fold = features.loc[
        train_mask,
        "target",
    ]

    X_valid_fold = features.loc[
        valid_mask,
        feature_cols,
    ]

    y_valid_fold = features.loc[
        valid_mask,
        "target",
    ]

    if model_name == "CatBoost":
        model = CatBoostClassifier(
            iterations=600,
            depth=6,
            learning_rate=0.03,
            loss_function="Logloss",
            random_seed=RANDOM_STATE,
            verbose=False,
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            eval_set=(X_valid_fold, y_valid_fold),
            early_stopping_rounds=80,
            verbose=False,
        )

        best_iteration = model.get_best_iteration()

    elif model_name == "HistGradientBoosting":
        model = HistGradientBoostingClassifier(
            learning_rate=0.045,
            max_iter=350,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )

        model.fit(
            X_train_fold,
            y_train_fold,
        )

        best_iteration = np.nan

    else:
        raise ValueError(
            "model_name должен быть CatBoost или HistGradientBoosting"
        )

    score = model.predict_proba(
        X_valid_fold
    )[:, 1]

    return {
        "model": model_name,
        "train_size": len(X_train_fold),
        "valid_size": len(X_valid_fold),
        "n_bots_valid": int(y_valid_fold.sum()),
        "p_at_r_70": precision_at_recall(
            y_valid_fold,
            score,
        ),
        "roc_auc": roc_auc_score(
            y_valid_fold,
            score,
        ),
        "pr_auc": average_precision_score(
            y_valid_fold,
            score,
        ),
        "best_iteration": best_iteration,
    }


In [13]:
folds = [
    ("2026-04-11", "2026-04-13"),
    ("2026-04-14", "2026-04-16"),
    ("2026-04-17", "2026-04-19"),
]

results = []

for model_name in [
    "CatBoost",
    "HistGradientBoosting",
]:
    for valid_start, valid_end in folds:
        result = evaluate_temporal_fold(
            valid_start,
            valid_end,
            train_features,
            feature_cols,
            model_name,
        )

        result["valid_start"] = valid_start
        result["valid_end"] = valid_end
        results.append(result)

cv_results = pd.DataFrame(results)

cv_results.sort_values(
    ["model", "valid_start"]
)


,model,train_size,valid_size,n_bots_valid,p_at_r_70,roc_auc,pr_auc,best_iteration,valid_start,valid_end
0,CatBoost,4217,2556,199,0.494700,0.884328,0.680170,544.0,2026-04-11,2026-04-13
1,CatBoost,6773,2367,195,0.578059,0.911248,0.723414,546.0,2026-04-14,2026-04-16
2,CatBoost,9140,1951,160,0.634831,0.897669,0.725027,564.0,2026-04-17,2026-04-19
3,HistGradientBoosting,4217,2556,199,0.472973,0.882819,0.677600,NaN,2026-04-11,2026-04-13
4,HistGradientBoosting,6773,2367,195,0.524904,0.902413,0.709329,NaN,2026-04-14,2026-04-16
5,HistGradientBoosting,9140,1951,160,0.541063,0.892748,0.717647,NaN,2026-04-17,2026-04-19


In [14]:
cv_summary = (
    cv_results
    .groupby("model")[
        [
            "p_at_r_70",
            "roc_auc",
            "pr_auc",
        ]
    ]
    .agg(["mean", "std"])
)

cv_summary


p_at_r_70             roc_auc              pr_auc  \
                          mean       std      mean       std      mean   
model                                                                    
CatBoost              0.569197  0.070485  0.897748  0.013460  0.709537   
HistGradientBoosting  0.512980  0.035577  0.892660  0.009797  0.701525   

                                
                           std  
model                           
CatBoost              0.025446  
HistGradientBoosting  0.021133

По временной проверке тоже стало лучше
Но все признаки пока описывают cookie отдельно. Следующая задача тогда: посмотреть, не повторяются ли одни и те же объявления, запросы, категории и User-Agent у разных куки
Для ботов это вполне может быть сигналом